In [37]:
import sys
import pandas as pd

sys.path.append("..")

from src.load import loadMatches, loadEvents, LEAKS

# loading match data and looking for shots before we clean

In [38]:
final = loadEvents(3869685)

shots = final[final["type"] == "Shot"].copy()

print("Total events:", len(final))
print("Total shots:", len(shots))

Total events: 4407
Total shots: 38


# here now i remvoed penalty shootout 

In [39]:
shootouts = (shots["period"] == 5).sum()

print("Shootout kicks:", shootouts)

shots = shots[shots["period"] != 5].copy()

print("Shots remaining:", len(shots))

Shootout kicks: 8
Shots remaining: 30


# removing in game penalities durng reg time 

In [40]:
penalties = (shots["shot_type"] == "Penalty").sum()

print("In-game penalties:", penalties)

shots = shots[shots["shot_type"] != "Penalty"].copy()

print("Shots remaining:", len(shots))

In-game penalties: 3
Shots remaining: 27


# now here we are crweating label for goal scored where 1 is Goal and 0 is no goal when shot

In [41]:
shots["goal"] = (shots["shot_outcome"] == "Goal").astype(int)

print(shots["goal"].value_counts())

goal
0    24
1     3
Name: count, dtype: int64


Checking for missing values 

In [42]:
flags = [
    "under_pressure",
    "shot_first_time",
    "shot_one_on_one",
    "shot_open_goal",
    "shot_aerial_won",
    "shot_follows_dribble",
    "shot_redirect"
]

for flag in flags:
    if flag in shots.columns:
        print(flag, ":", shots[flag].isna().sum())
    else:
        print(flag, ": column missing")

under_pressure : 26
shot_first_time : 16
shot_one_on_one : 25
shot_open_goal : column missing
shot_aerial_won : 26
shot_follows_dribble : column missing
shot_redirect : column missing


# cleanig the flag clouns 

In [43]:
for flag in flags:
    if flag not in shots.columns:
        shots[flag] = 0
    else:
        shots[flag] = shots[flag].fillna(False).astype(int)

print(shots[flags].isna().sum())

under_pressure          0
shot_first_time         0
shot_one_on_one         0
shot_open_goal          0
shot_aerial_won         0
shot_follows_dribble    0
shot_redirect           0
dtype: int64


we are now removing the datas leakge colusn 

In [44]:
foundLeaks = [col for col in LEAKS if col in shots.columns]

print("Leak columns found:", foundLeaks)

shots = shots.drop(columns=LEAKS, errors="ignore")

print("Leak columns remaining:", len(set(LEAKS) & set(shots.columns)))

Leak columns found: ['shot_outcome', 'shot_statsbomb_xg', 'shot_end_location', 'out', 'pass_outcome', 'pass_recipient', 'pass_goal_assist', 'pass_shot_assist', 'pass_assisted_shot_id', 'pass_assisted_shot_id', 'pass_recipient_id', 'pass_deflected']
Leak columns remaining: 0


now we are extraxting shots cooordinates 

In [45]:
shots["x"] = shots["location"].apply(lambda loc: loc[0])
shots["y"] = shots["location"].apply(lambda loc: loc[1])

print(shots[["x", "y"]].head())

          x     y
4207   92.4  30.0
4208   99.2  47.9
4209  103.1  34.6
4211   94.3  23.5
4212  111.8  32.1


In [46]:
columns = [
    "id", "match_id", "period", "minute", "second",
    "team", "player", "position", "x", "y",
    "play_pattern", "shot_body_part", "shot_technique",
    "shot_type", "shot_key_pass_id",
    "under_pressure", "shot_first_time", "shot_one_on_one",
    "shot_open_goal", "shot_aerial_won",
    "shot_follows_dribble", "shot_redirect",
    "shot_freeze_frame", "goal"
]

missing = [col for col in columns if col not in shots.columns]

print("Missing columns:", missing)

Missing columns: []


In [47]:
shots = shots[columns].copy()

print("Total shots:", len(shots))
print("Total columns:", len(shots.columns))
print(shots.head())

Total shots: 27
Total columns: 24
                                        id  match_id  period  minute  second  \
4207  545c2c84-018f-4570-a01c-753823feaeac   3869685       1       4      40   
4208  4ad26294-8aaf-4d69-83dd-bbf9ef797b32   3869685       1       7      36   
4209  6d498191-05f0-432c-8764-03aea4ef9fb8   3869685       1      16      15   
4211  f227a92e-d86c-4f3d-aa75-5ccb71adcae6   3869685       1      31      35   
4212  ef86f4d9-7acd-4ed0-a5ec-9129079e8fbe   3869685       1      35      22   

           team                           player               position  \
4207  Argentina              Alexis Mac Allister   Left Center Midfield   
4208  Argentina           Rodrigo Javier De Paul  Right Center Midfield   
4209  Argentina  Ángel Fabián Di María Hernández              Left Wing   
4211  Argentina              Alexis Mac Allister   Left Center Midfield   
4212  Argentina  Ángel Fabián Di María Hernández              Left Wing   

          x     y  ...            

her we are verifying the freezed frmaes 

In [48]:
missingFrames = shots["shot_freeze_frame"].isna().sum()

print("Total shots:", len(shots))
print("Shots without freeze frames:", missingFrames)
print("First freeze frame type:", type(shots["shot_freeze_frame"].iloc[0]))

Total shots: 27
Shots without freeze frames: 0
First freeze frame type: <class 'list'>


# creating cleanshot fun for resue

In [49]:
def CleanShots(events):
    """Clean shot events by removing penalties, leaks, and missing flags."""

    shots = events[events["type"] == "Shot"].copy()

    # Shootouts and in-game penalties are excluded from the goal model.
    shots = shots[shots["period"] != 5].copy()
    shots = shots[shots["shot_type"] != "Penalty"].copy()

    shots["goal"] = (shots["shot_outcome"] == "Goal").astype(int)

    for flag in flags:
        if flag not in shots.columns:
            shots[flag] = 0
        else:
            shots[flag] = shots[flag].fillna(False).astype(int)

    shots = shots.drop(columns=LEAKS, errors="ignore")

    shots["x"] = shots["location"].apply(lambda loc: loc[0])
    shots["y"] = shots["location"].apply(lambda loc: loc[1])

    shots = shots[columns].copy()

    return shots

#testing the finctiopn

In [50]:
testShots = CleanShots(final)

print("Total shots:", len(testShots))
print("Total columns:", len(testShots.columns))
print("Goals:", testShots["goal"].sum())
print("Missing freeze frames:", testShots["shot_freeze_frame"].isna().sum())

Total shots: 27
Total columns: 24
Goals: 3
Missing freeze frames: 0


cheking full match list before we start

In [51]:
matches = loadMatches()

print("Total matches:", len(matches))
print("Unique match IDs:", matches["match_id"].nunique())
print("First five match IDs:", matches["match_id"].head().tolist())

Total matches: 3961
Unique match IDs: 3961
First five match IDs: [3895292, 3895320, 3895158, 3895107, 3895340]


In [52]:
matchId = matches["match_id"].iloc[0]
events = loadEvents(matchId)
clean = CleanShots(events)

print("Match ID:", matchId)
print("Cleaned shots:", len(clean))
print("Columns:", len(clean.columns))

Match ID: 3895292
Cleaned shots: 28
Columns: 24


In [53]:
from pathlib import Path

batchFolder = Path("../data/clean/batches")
batchFolder.mkdir(parents=True, exist_ok=True)

path = batchFolder / "shots_0000.parquet"

if path.exists():
    print("First batch already saved. Skipping.")
else:
    batchShots = []

    for matchId in matches["match_id"].head(100):
        events = loadEvents(matchId)
        clean = CleanShots(events)
        batchShots.append(clean)

    batch = pd.concat(batchShots, ignore_index=True)
    batch.to_parquet(path, index=False)

    print("Matches processed:", 100)
    print("Shots saved:", len(batch))
    print("File exists:", path.exists())

First batch already saved. Skipping.


In [54]:
saved = pd.read_parquet(batchFolder / "shots_0000.parquet")

print("Saved shots:", len(saved))
print("Saved columns:", len(saved.columns))
print("Missing freeze frames:", saved["shot_freeze_frame"].isna().sum())
print("Freeze frame type:", type(saved["shot_freeze_frame"].iloc[0]))

Saved shots: 2453
Saved columns: 24
Missing freeze frames: 0
Freeze frame type: <class 'numpy.ndarray'>


In [55]:
import gc

for start in range(100, len(matches), 100):
    end = min(start + 100, len(matches))
    path = batchFolder / f"shots_{start:04d}.parquet"

    if path.exists():
        print("Already saved:", start, "to", end)
        continue

    batchShots = []

    for matchId in matches["match_id"].iloc[start:end]:
        events = loadEvents(matchId)
        clean = CleanShots(events)
        batchShots.append(clean)

    batch = pd.concat(batchShots, ignore_index=True)
    batch.to_parquet(path, index=False)

    print("Saved matches:", start, "to", end, "| Shots:", len(batch))

    del batchShots, batch, clean, events
    gc.collect()

Already saved: 100 to 200
Already saved: 200 to 300
Already saved: 300 to 400
Already saved: 400 to 500
Already saved: 500 to 600
Already saved: 600 to 700
Already saved: 700 to 800
Already saved: 800 to 900
Already saved: 900 to 1000
Already saved: 1000 to 1100
Already saved: 1100 to 1200
Already saved: 1200 to 1300
Already saved: 1300 to 1400
Already saved: 1400 to 1500
Already saved: 1500 to 1600
Already saved: 1600 to 1700
Already saved: 1700 to 1800
Already saved: 1800 to 1900
Already saved: 1900 to 2000
Already saved: 2000 to 2100
Already saved: 2100 to 2200
Already saved: 2200 to 2300
Already saved: 2300 to 2400
Already saved: 2400 to 2500
Already saved: 2500 to 2600
Already saved: 2600 to 2700
Already saved: 2700 to 2800
Already saved: 2800 to 2900
Already saved: 2900 to 3000
Already saved: 3000 to 3100
Already saved: 3100 to 3200
Already saved: 3200 to 3300
Already saved: 3300 to 3400
Already saved: 3400 to 3500
Already saved: 3500 to 3600
Already saved: 3600 to 3700
Already s

In [56]:
import pyarrow.parquet as pq

batchFiles = sorted(batchFolder.glob("shots_*.parquet"))

totalShots = 0

for file in batchFiles:
    info = pq.read_metadata(file)
    totalShots += info.num_rows

print("Total batches:", len(batchFiles))
print("Total cleaned shots:", totalShots)
print("Matches processed:", len(matches))

Total batches: 40
Total cleaned shots: 99746
Matches processed: 3961


In [69]:
import pyarrow.parquet as pq

output = batchFolder.parent / "shots.parquet"

ExpectedColumns = set(columns)

if output.exists():
    FileInfo = pq.ParquetFile(output)
    ValidRows = FileInfo.metadata.num_rows == 99746
    ValidColumns = set(FileInfo.schema_arrow.names) == ExpectedColumns
else:
    ValidRows = False
    ValidColumns = False

if ValidRows and ValidColumns:
    print("Final dataset already saved. Skipping.")

else:
    writer = None

    try:
        for path in batchFiles:
            table = pq.read_table(path)

            if writer is None:
                writer = pq.ParquetWriter(output, table.schema)

            writer.write_table(table)
            del table

    finally:
        if writer is not None:
            writer.close()

print("File saved:", output.exists())
print("Total shots:", pq.read_metadata(output).num_rows)

Final dataset already saved. Skipping.
File saved: True
Total shots: 99746


In [58]:
import pyarrow.parquet as pq

file = pq.ParquetFile("../data/clean/shots.parquet")

print("Total rows:", file.metadata.num_rows)
print("Total columns:", file.metadata.num_columns)
print("Columns:", file.schema_arrow.names)

Total rows: 99746
Total columns: 29
Columns: ['id', 'match_id', 'period', 'minute', 'second', 'team', 'player', 'position', 'x', 'y', 'play_pattern', 'shot_body_part', 'shot_technique', 'shot_type', 'shot_key_pass_id', 'under_pressure', 'shot_first_time', 'shot_one_on_one', 'shot_open_goal', 'shot_aerial_won', 'shot_follows_dribble', 'shot_redirect', 'shot_freeze_frame', 'goal']


In [59]:
print("Top-level columns:", len(file.schema_arrow.names))
print("Parquet physical columns:", file.metadata.num_columns)
print("Freeze frame type:", file.schema_arrow.field("shot_freeze_frame").type)

Top-level columns: 24
Parquet physical columns: 29
Freeze frame type: list<element: struct<location: list<element: double>, player: struct<id: int64, name: string>, position: struct<id: int64, name: string>, teammate: bool>>


checking for missing frames now 

In [60]:
missingFrames = 0

for batch in file.iter_batches(batch_size=1000, columns=["shot_freeze_frame"]):
    frame = batch.to_pandas()
    missingFrames += frame["shot_freeze_frame"].isna().sum()

print("Shots without freeze frames:", missingFrames)

Shots without freeze frames: 0


now i am here verify the goals labe 

In [61]:
goalCounts = {0: 0, 1: 0}
missingGoals = 0

for batch in file.iter_batches(batch_size=1000, columns=["goal"]):
    goals = batch.to_pandas()["goal"]

    missingGoals += goals.isna().sum()

    for value in [0, 1]:
        goalCounts[value] += (goals == value).sum()

print("Non-goals:", goalCounts[0])
print("Goals:", goalCounts[1])
print("Missing labels:", missingGoals)
print("Total:", sum(goalCounts.values()))

Non-goals: 89513
Goals: 10233
Missing labels: 0
Total: 99746


checking for any data leaskge 

In [62]:
remainingLeaks = [col for col in LEAKS if col in file.schema_arrow.names]

print("Remaining leakage columns:", remainingLeaks)

Remaining leakage columns: []


In [63]:
for flag in flags:
    values = set()
    missing = 0

    for batch in file.iter_batches(batch_size=1000, columns=[flag]):
        data = batch.to_pandas()[flag]

        values.update(data.dropna().unique().tolist())
        missing += data.isna().sum()

    print(flag, "| Values:", sorted(values), "| Missing:", missing)

under_pressure | Values: [0, 1] | Missing: 0
shot_first_time | Values: [0, 1] | Missing: 0
shot_one_on_one | Values: [0, 1] | Missing: 0
shot_open_goal | Values: [0, 1] | Missing: 0
shot_aerial_won | Values: [0, 1] | Missing: 0
shot_follows_dribble | Values: [0, 1] | Missing: 0
shot_redirect | Values: [0, 1] | Missing: 0


Checking fdor duplicate shoyts 

In [64]:
shotIds = set()
duplicates = 0

for batch in file.iter_batches(batch_size=1000, columns=["id"]):
    ids = batch.to_pandas()["id"]

    for shotId in ids:
        if shotId in shotIds:
            duplicates += 1
        else:
            shotIds.add(shotId)

print("Unique shot IDs:", len(shotIds))
print("Duplicate shots:", duplicates)

Unique shot IDs: 99746
Duplicate shots: 0


In [65]:
TotalShots = 0
ShootoutCount = 0
PenaltyCount = 0

for MatchId in matches["match_id"]:
    Events = loadEvents(MatchId)
    Shots = Events[Events["type"] == "Shot"]

    TotalShots += len(Shots)

    Shootouts = Shots[Shots["period"] == 5]
    ShootoutCount += len(Shootouts)

    Shots = Shots[Shots["period"] != 5]
    Penalties = Shots[Shots["shot_type"] == "Penalty"]
    PenaltyCount += len(Penalties)

print("Total original shots:", TotalShots)
print("Shootout kicks removed:", ShootoutCount)
print("In-game penalties removed:", PenaltyCount)
print("Cleaned shots:", TotalShots - ShootoutCount - PenaltyCount)

Total original shots: 101227
Shootout kicks removed: 363
In-game penalties removed: 1118
Cleaned shots: 99746


In [66]:
OriginalShots = final[final["type"] == "Shot"].copy()

print("Original shots:", len(OriginalShots))
print("Rows remaining after dropna():", len(OriginalShots.dropna()))

Original shots: 38
Rows remaining after dropna(): 0


In [67]:
MissingFlags = {Flag: 0 for Flag in flags}

for MatchId in matches["match_id"]:
    Events = loadEvents(MatchId)
    Shots = Events[Events["type"] == "Shot"].copy()

    Shots = Shots[Shots["period"] != 5]
    Shots = Shots[Shots["shot_type"] != "Penalty"]

    for Flag in flags:
        if Flag in Shots.columns:
            MissingFlags[Flag] += Shots[Flag].isna().sum()
        else:
            MissingFlags[Flag] += len(Shots)

for Flag, Count in MissingFlags.items():
    print(Flag, ":", Count)

under_pressure : 74777
shot_first_time : 68502
shot_one_on_one : 94815
shot_open_goal : 98694
shot_aerial_won : 91247
shot_follows_dribble : 99682
shot_redirect : 99548


In [68]:
FoundLeaks = set()

for MatchId in matches["match_id"]:
    Events = loadEvents(MatchId)

    for Column in LEAKS:
        if Column in Events.columns:
            FoundLeaks.add(Column)

print("Leak columns found:", sorted(FoundLeaks))
print("Total unique leak columns:", len(FoundLeaks))

Leak columns found: ['out', 'pass_assisted_shot_id', 'pass_deflected', 'pass_goal_assist', 'pass_outcome', 'pass_recipient', 'pass_recipient_id', 'pass_shot_assist', 'shot_deflected', 'shot_end_location', 'shot_outcome', 'shot_saved_off_target', 'shot_saved_to_post', 'shot_statsbomb_xg']
Total unique leak columns: 14
